<a href="https://colab.research.google.com/github/e23383/Statistical-Learning-e23383/blob/main/Assignment_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy import linalg
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import EmpiricalCovariance
import warnings
warnings.filterwarnings('ignore')

!pip install factor-analyzer -q

from factor_analyzer import FactorAnalyzer

np.random.seed(42)
n_samples = 5000
n_features = 3

base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

base_data[4000:, 0] += 0.015
base_data[4000:, 2] -= 0.010

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    n_samples = len(df)
    m = df.shape[1]

    chunk_size = n_samples // g_chunks
    chunks = []
    for i in range(g_chunks):
        start_idx = i * chunk_size
        end_idx = (i + 1) * chunk_size if i < g_chunks - 1 else n_samples
        chunks.append(df.iloc[start_idx:end_idx].values)

    global_mean = df.values.mean(axis=0)
    chunk_means = np.array([chunk.mean(axis=0) for chunk in chunks])
    chunk_sizes = np.array([len(chunk) for chunk in chunks])

    W = np.zeros((m, m))
    B = np.zeros((m, m))

    for idx, chunk in enumerate(chunks):
        centered = chunk - chunk_means[idx]
        W += centered.T @ centered
        mean_diff = (chunk_means[idx] - global_mean).reshape(-1, 1)
        B += chunk_sizes[idx] * (mean_diff @ mean_diff.T)

    Lambda_val = np.linalg.det(W) / np.linalg.det(B + W)

    df_chi2 = m * (g_chunks - 1)
    bartlett_stat = - (n_samples - 1 - (m + g_chunks) / 2) * np.log(Lambda_val)
    p_value = 1 - stats.chi2.cdf(bartlett_stat, df_chi2)

    return {
        'wilks_lambda': Lambda_val,
        'bartlett_chi2': bartlett_stat,
        'df': df_chi2,
        'p_value': p_value,
        'alpha': 0.05,
        'conclusion': 'Reject H0: First moment is NOT homogeneous (shift detected)'
                      if p_value < 0.05 else 'Fail to reject H0: First moment is homogeneous'
    }

results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
print("=" * 60)
print("MANOVA RESULTS FOR FIRST MOMENT HOMOGENEITY")
print("=" * 60)
for key, value in results.items():
    print(f"{key}: {value}")
print("=" * 60)

class PCADiagnostics:
    def __init__(self, data, n_components=4):
        self.data = data.values if isinstance(data, pd.DataFrame) else data
        self.n_samples, self.m = self.data.shape
        self.n_components = min(n_components, self.m)
        self._fit()

    def _fit(self):
        self.scaler = StandardScaler()
        self.X_scaled = self.scaler.fit_transform(self.data)
        self.cov_matrix = np.cov(self.X_scaled.T)
        eigenvalues, eigenvectors = np.linalg.eigh(self.cov_matrix)
        idx = np.argsort(eigenvalues)[::-1]
        self.eigenvalues = eigenvalues[idx]
        self.eigenvectors = eigenvectors[:, idx]
        self.scores = self.X_scaled @ self.eigenvectors

    def compute_metrics_for_k(self, k):
        T2 = np.zeros(self.n_samples)
        Q = np.zeros(self.n_samples)

        for i in range(self.n_samples):
            T2[i] = np.sum((self.scores[i, :k] ** 2) / self.eigenvalues[:k])
            reconstruction = self.scores[i, :k] @ self.eigenvectors[:, :k].T
            residual = self.X_scaled[i] - reconstruction
            Q[i] = np.sum(residual ** 2)

        return {
            'k': k,
            'mean_T2': np.mean(T2),
            'mean_Q': np.mean(Q),
            'var_explained': np.sum(self.eigenvalues[:k]) / np.sum(self.eigenvalues) * 100
        }

    def get_all_metrics(self):
        metrics = []
        for k in range(1, self.m + 1):
            metrics.append(self.compute_metrics_for_k(k))
        return metrics

class FactorAnalysisEngine:
    def __init__(self, data, n_factors=2, rotation='varimax'):
        self.data = data.values if isinstance(data, pd.DataFrame) else data
        self.n_samples, self.m = self.data.shape
        self.n_factors = n_factors
        self.rotation = rotation

        assert self.n_samples > self.m, f"n_samples ({self.n_samples}) must be > m ({self.m})"
        assert self.n_factors < self.m, f"n_factors ({n_factors}) must be < m ({self.m})"

        self._fit()

    def _fit(self):
        self.mean = np.mean(self.data, axis=0)
        self.std = np.std(self.data, axis=0, ddof=1)
        self.std = np.maximum(self.std, 1e-15)
        self.Z = (self.data - self.mean) / self.std

        self.fa = FactorAnalyzer(n_factors=self.n_factors, rotation=self.rotation)
        self.fa.fit(self.Z)

        self.loadings = self.fa.loadings_
        self.communalities = np.sum(self.loadings ** 2, axis=1)
        self.uniquenesses = 1 - self.communalities

        R = np.corrcoef(self.Z.T)
        R_inv = np.linalg.pinv(R)
        self.factor_scores = self.Z @ R_inv @ self.loadings
        self.factor_variances = np.var(self.factor_scores, axis=0, ddof=1)

    def get_diagnostics(self):
        return {
            'communalities': self.communalities,
            'uniquenesses': self.uniquenesses,
            'loadings': self.loadings,
            'factor_scores': self.factor_scores,
            'factor_variances': self.factor_variances,
            'avg_communality_pct': np.mean(self.communalities) * 100,
            'avg_uniqueness_pct': np.mean(self.uniquenesses) * 100
        }

np.random.seed(42)
n_samples = 2500

f1 = np.random.normal(0, 1, n_samples)
f2 = np.random.normal(0, 1, n_samples)

s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

df_asset = pd.DataFrame(
    data=np.vstack([s1, s2, s3, s4]).T,
    columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4']
)

print("\n" + "=" * 60)
print("PCA DIAGNOSTICS RESULTS")
print("=" * 60)

pca_diag = PCADiagnostics(df_asset)
pca_metrics = pca_diag.get_all_metrics()

print("\nPrincipal Component Eigenvalues:")
for i, ev in enumerate(pca_diag.eigenvalues, 1):
    print(f"PC{i}: {ev:.4f}")

print(f"\nTotal Variance: {np.sum(pca_diag.eigenvalues):.4f}")
print(f"Tr(Covariance): {np.trace(pca_diag.cov_matrix):.4f}")

print("\nTruncation Analysis:")
for metric in pca_metrics:
    print(f"k={metric['k']}: Var Explained={metric['var_explained']:.2f}%, Mean T²={metric['mean_T2']:.4f}, Mean Q={metric['mean_Q']:.4f}")

print("\n" + "=" * 60)
print("FACTOR ANALYSIS DIAGNOSTICS RESULTS")
print("=" * 60)

fa_engine = FactorAnalysisEngine(df_asset, n_factors=2, rotation='varimax')
fa_results = fa_engine.get_diagnostics()

print(f"\nAverage System Communality %: {fa_results['avg_communality_pct']:.2f}%")
print(f"Average System Uniqueness %: {fa_results['avg_uniqueness_pct']:.2f}%")

print("\nPer-Sensor Communality & Uniqueness:")
sensor_names = ['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4']
for i, name in enumerate(sensor_names):
    print(f"{name}: h²={fa_results['communalities'][i]:.4f}, φ²={fa_results['uniquenesses'][i]:.4f}")

print("\nRotated Factor Loadings Matrix:")
for i, name in enumerate(sensor_names):
    print(f"{name}: Factor1={fa_results['loadings'][i,0]:.4f}, Factor2={fa_results['loadings'][i,1]:.4f}")

print(f"\nLatent Factor Empirical Variances: F1={fa_results['factor_variances'][0]:.4f}, F2={fa_results['factor_variances'][1]:.4f}")

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Feature Loadings Heatmap', 'Absolute Eigenvalues', 'Variance Explained',
                    'Residual Unexplained Variance', 'Mean Hotelling\'s T²', 'Mean Q Statistic (SPE)'),
    horizontal_spacing=0.15,
    vertical_spacing=0.2
)

loadings_abs = np.abs(pca_diag.eigenvectors[:, :pca_diag.m])
fig.add_trace(
    go.Heatmap(z=loadings_abs, x=[f'PC{i+1}' for i in range(pca_diag.m)],
               y=sensor_names, colorscale='YlOrRd', zmin=0, zmax=1),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=[f'PC{i+1}' for i in range(pca_diag.m)], y=pca_diag.eigenvalues,
           marker_color='steelblue'),
    row=1, col=2
)

var_exp = pca_diag.eigenvalues / np.sum(pca_diag.eigenvalues) * 100
cum_var_exp = np.cumsum(var_exp)
fig.add_trace(
    go.Bar(x=[f'PC{i+1}' for i in range(pca_diag.m)], y=var_exp,
           name='Marginal', marker_color='lightblue'),
    row=1, col=3
)
fig.add_trace(
    go.Scatter(x=[f'PC{i+1}' for i in range(pca_diag.m)], y=cum_var_exp,
               name='Cumulative', line=dict(dash='dash', color='red', width=2),
               mode='lines+markers'),
    row=1, col=3
)

residual_var = 100 - cum_var_exp
fig.add_trace(
    go.Bar(x=[f'k={i+1}' for i in range(pca_diag.m)], y=residual_var,
           marker_color='coral'),
    row=2, col=1
)

k_values = [m['k'] for m in pca_metrics]
t2_values = [m['mean_T2'] for m in pca_metrics]
fig.add_trace(
    go.Scatter(x=k_values, y=t2_values, mode='lines+markers',
               marker=dict(size=10, color='darkgreen'),
               line=dict(width=2, color='green')),
    row=2, col=2
)

q_values = [m['mean_Q'] for m in pca_metrics]
fig.add_trace(
    go.Scatter(x=k_values, y=q_values, mode='lines+markers',
               marker=dict(size=10, color='darkred'),
               line=dict(width=2, color='red')),
    row=2, col=3
)

fig.update_layout(height=800, width=1400, showlegend=True,
                  title_text="PCA Diagnostics Dashboard", title_x=0.5)
fig.show()

fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Structural Loadings Matrix (Rotated)',
                    'Variance Allocation: Communality vs Uniqueness',
                    'Sensor Uniqueness Noise Floor Profile',
                    'Latent Factor Empirical Variance'),
    horizontal_spacing=0.2,
    vertical_spacing=0.25
)

loadings_rot_abs = np.abs(fa_results['loadings'])
fig2.add_trace(
    go.Heatmap(z=loadings_rot_abs, x=['Factor 1', 'Factor 2'],
               y=sensor_names, colorscale='YlOrRd', zmin=0, zmax=1),
    row=1, col=1
)

communality_pct = fa_results['communalities'] * 100
uniqueness_pct = fa_results['uniquenesses'] * 100

for i, name in enumerate(sensor_names):
    fig2.add_trace(
        go.Bar(name=name, x=[name], y=[communality_pct[i]],
               marker_color='#1f77b4', legendgroup=name, showlegend=False),
        row=1, col=2
    )
    fig2.add_trace(
        go.Bar(name=name, x=[name], y=[uniqueness_pct[i]],
               marker_color='#ff7f0e', legendgroup=name, showlegend=False),
        row=1, col=2
    )

fig2.add_trace(
    go.Scatter(x=sensor_names, y=fa_results['uniquenesses'],
               mode='lines+markers', line=dict(dash='dot', color='#d62728', width=2),
               marker=dict(symbol='x', size=10, color='#d62728')),
    row=2, col=1
)

fig2.add_trace(
    go.Bar(x=['Factor 1', 'Factor 2'], y=fa_results['factor_variances'],
           marker_color='#2ca02c', marker_line=dict(color='black', width=0.5)),
    row=2, col=2
)

fig2.update_layout(height=800, width=1200, showlegend=False,
                  title_text="Factor Analysis Diagnostics Dashboard", title_x=0.5)
fig2.update_xaxes(tickangle=25, row=2, col=1)
fig2.update_xaxes(title_text="Sensor Channel", row=2, col=1)
fig2.update_yaxes(title_text="Uniqueness (φ²)", row=2, col=1)
fig2.update_yaxes(title_text="Empirical Variance", row=2, col=2)

fig2.show()

print("\n" + "=" * 60)
print("ANSWERS TO CONCEPTUAL QUESTIONS")
print("=" * 60)

print("\nQUESTION 1: The Total Variance Illusion")
print("-" * 40)
print("1. Uniqueness near 100% indicates Sensor_4 contains almost entirely independent")
print("   noise with virtually no shared variance with other sensors.")
print("\n2. PCA maximizes total variance without distinguishing signal from noise.")
print("   Sensor_4's large variance (~2.0) dominates the eigenvalue calculation,")
print("   forcing PC1 to align with this noisy sensor despite its lack of correlation.")
print("\n3. Risk: PCA may treat sensor noise as important signal, causing false alarms")
print("   or missing real structural changes masked by the noise floor.")

print("\nQUESTION 2: Decoupling Structural Loading via Rotation")
print("-" * 40)
print("1. Varimax rotation simplifies loadings by maximizing variance of squared")
print("   loadings per factor, creating 'simple structure' where each sensor loads")
print("   strongly on only one factor. Unlike PCA's hierarchical variance capture,")
print("   rotation redistributes explained variance across factors for interpretability.")
print("\n2. From an operator standpoint: The rotated FA heatmap shows clean groupings")
print("   (Sensors 1-2 on Factor 1, Sensor 3 on Factor 2), making fault isolation")
print("   intuitive. PCA eigenvectors mix all sensors across components, obscuring")
print("   which physical process is failing.")

print("\nQUESTION 3: Determining Subspace Truncation (k) using T² and Q Profiles")
print("-" * 40)
print("1. Mean Q drops sharply from k=1 to k=2 (residual variance plummets as")
print("   second physical factor is captured), then flattens from k=2 to k=3")
print("   (remaining residual is true white noise).")
print("\n2. The sharp drop identifies k=2 as the true hidden dimensionality because")
print("   Q measures unexplained variance. When Q stops decreasing substantially,")
print("   remaining PCs explain only noise, not signal.")
print("\n3. Choosing k=3 forces pure noise into the 'clean' subspace, inflating")
print("   T² statistics and causing false alarms when noise naturally varies.")

print("\nQUESTION 4: Operational Trade-offs in System Health Monitoring")
print("-" * 40)
print("FA Strategy is MORE robust against single sensor failure.")
print("\nWhy: Sensor_4's uniqueness (φ² ≈ 98%) in FA results reveals it carries")
print("almost no shared structural information. FA can ignore this channel's")
print("noise floor when monitoring latent factors, while PCA would treat its")
print("large variance as a principal component, triggering false alarms.")
print("\nFA's explicit separation of communality vs. uniqueness provides a")
print("mathematical guardrail against individual sensor degradation.")
print("=" * 60)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 254.9 kB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
MANOVA RESULTS FOR FIRST MOMENT HOMOGENEITY
wilks_lambda: 0.9904519796929591
bartlett_chi2: 47.9215049961488
df: 12
p_value: 3.2255259508895406e-06
alpha: 0.05
conclusion: Reject H0: First moment is NOT homogeneous (shift detected)

PCA DIAGNOSTICS RESULTS

Principal Component Eigenvalues:
PC1: 1.9752
PC2: 1.0074
PC3: 0.8814
PC4: 0.1376

Total Variance: 4.0016
Tr(Covariance): 4.0016

Truncation Analysis:
k=1: Var Explained=49.36%, Mean T²=0.9996, Mean Q=2.0256
k=2: Var Explained=74.53%, Mean T²=1.9992, Mean Q=1.0186
k=3: Var Explained=96.56%, Mean T²=2.9988, Mean Q=0.1375
k=4: Var Explained=100.00%, Mean T²=3.9984, Mean Q=0.0000

FACTOR ANALYSIS DIAGNOSTICS RESULTS

Average System Communality %: 68.07%
Average System Uniqueness %: 31.93%

Per-Sensor Communality & Uniqueness


ANSWERS TO CONCEPTUAL QUESTIONS

QUESTION 1: The Total Variance Illusion
----------------------------------------
1. Uniqueness near 100% indicates Sensor_4 contains almost entirely independent
   noise with virtually no shared variance with other sensors.

2. PCA maximizes total variance without distinguishing signal from noise.
   Sensor_4's large variance (~2.0) dominates the eigenvalue calculation,
   forcing PC1 to align with this noisy sensor despite its lack of correlation.

3. Risk: PCA may treat sensor noise as important signal, causing false alarms
   or missing real structural changes masked by the noise floor.

QUESTION 2: Decoupling Structural Loading via Rotation
----------------------------------------
1. Varimax rotation simplifies loadings by maximizing variance of squared
   loadings per factor, creating 'simple structure' where each sensor loads
   strongly on only one factor. Unlike PCA's hierarchical variance capture,
   rotation redistributes explained variance a